In [1]:
import os
import yaml
import shutil
from ultralytics import YOLO
from roboflow import Roboflow
import glob

# CLEAN OLD COMBINED FOLDER AND DOWNLOADED DATASETS
if os.path.exists("combined"):
    shutil.rmtree("combined")
    print("Removed previous 'combined' folder.\n")

# CLEANUP of potentially empty/corrupt downloaded folders (Datasets 2-5)
datasets_to_clean = [
    "Accessible-Toilets-1", "bathroom-2", "Door-handle-1", "Disabled-Bathroom-1", "sink-2"
]
for ds_name in datasets_to_clean:
    if os.path.exists(ds_name):
        shutil.rmtree(ds_name)
        print(f"Removed previous corrupt folder: {ds_name}")


# DOWNLOAD DATASETS AND DEFINE PATHS
rf = Roboflow(api_key="BJDNkOnpT2z6XuaFpI94")

# Download datasets
ds1 = rf.workspace("image-bounding-for-datasets").project("accessible-toilets").version(1).download("yolov8")
ds2 = rf.workspace("adrian-gorcea").project("bathroom-sdsne").version(2).download("yolov8")
ds3 = rf.workspace("new-workspace-vjwug").project("door-handle-hzojf").version(1).download("yolov8")
ds4 = rf.workspace("laimii").project("disabled-bathroom").version(1).download("yolov8")
ds5 = rf.workspace("cups-yjhli").project("sink-hhipp").version(2).download("yolov8")

print("Datasets downloaded.\n")

# Define downloaded locations
dataset_paths = [ds1.location, ds2.location, ds3.location, ds4.location, ds5.location]


# MAPPINGS (TARGET: 0..8)
dataset1_map = { # Original Names: ['grab_handle', 'toilet', 'wheelchair_logo']
    0: 1,    # grab_handle (Target 1)
    1: 0,    # toilet (Target 0)
    2: 6     # wheelchair_logo (Target 6)
}

dataset2_map = { # Original Names: ['mirror', 'sink', 'soap', 'towel', 'towel-sink-soap-mirror-wc', 'wc']
    0: 2,    # mirror (Target 2)
    1: 3,    # sink (Target 3) 
    2: 4,    # soap (Target 4)
    3: 5,    # towel (Target 5) 
    4: None, # junk/mixed class – drop boxes
    5: 0     # wc (Target 0: toilet) 
}

dataset3_map = { # Original Names: ['Door-handle'] (ID 0)
    0: 7     # Door-handle (Target 7)
}

dataset4_map = { # Original Names: ['Toilet', 'accessible-toilet-sign', 'emergency-button', 'grab-bars']
    0: 0,    # Toilet (Target 0) 
    1: 6,    # accessible-toilet-sign (Target 6: wheelchair_logo) 
    2: 8,    # emergency-button (Target 8) 
    3: 1,    # grab-bars (Target 1: grab_handle) 
}

dataset5_map = { # Original Names: ['0'] (ID 0, likely sink)
    0: 3      # '0' class (Target 3: sink)
}

dataset_maps = [dataset1_map, dataset2_map, dataset3_map, dataset4_map, dataset5_map]


# 3 LABEL REWRITE (CLASS REMAPPING & SEGMENTATION TRUNCATION)
def rewrite_label_file(path, id_map):
    """Safely remap classes, truncate to bounding box (5 elements), and delete empty files."""
    new_lines = []

    try:
        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                
                # Input validation
                try:
                    old_cls = int(parts[0])
                except ValueError:
                    continue

                if old_cls not in id_map or id_map[old_cls] is None:
                    continue

                # Rewrite class ID and truncate segmentation data
                parts[0] = str(id_map[old_cls])
                if len(parts) > 5:
                    parts = parts[:5]
                
                new_lines.append(" ".join(parts))

        if not new_lines:
            os.remove(path)
            return False

        with open(path, "w") as f:
            f.write("\n".join(new_lines) + "\n") 
        return True
    
    except Exception as e:
        print(f"ERROR processing label file {path}: {e}")
        return False


def rewrite_dataset_labels(base_path, id_map):
    """Uses recursive os.walk to find 'labels' folders and rewrites contents."""
    base_name = os.path.basename(base_path)
    print(f"\nRewriting labels for dataset: {base_name}")
    
    # Use os.walk for robust path finding
    for root, _, files in os.walk(base_path):
        if os.path.basename(root) == "labels":
            split_name = os.path.basename(os.path.dirname(root))
            if split_name in ["train", "valid", "test"]:
                files_to_process = [f for f in files if f.endswith(".txt")]
                print(f"  Found {len(files_to_process)} label files in '{split_name}/labels'.")
                
                for file in files_to_process:
                    rewrite_label_file(os.path.join(root, file), id_map)

# Apply mappings to each dataset
for base_path, id_map in zip(dataset_paths, dataset_maps):
    rewrite_dataset_labels(base_path, id_map)


# MERGE DATASETS SAFELY (CANONICAL YOLOv8 STRUCTURE)
# Create target merge directories
for split in ["train", "valid", "test"]:
    split_dir = os.path.join("combined", split)
    os.makedirs(os.path.join(split_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(split_dir, "labels"), exist_ok=True)


def merge_dataset(ds_index, ds_path):
    """Merges files, ensuring unique filenames (ds{index}_{split}_{stem})."""
    for root, _, files in os.walk(ds_path):
        if os.path.basename(root) != "labels":
            continue
        
        lbl_dir = root
        split = os.path.basename(os.path.dirname(lbl_dir))

        if split not in ["train", "valid", "test"]:
            continue 

        img_dir = os.path.join(os.path.dirname(lbl_dir), "images")
        
        if not os.path.exists(img_dir):
            continue
            
        for lbl_file_path in glob.glob(os.path.join(lbl_dir, "*.txt")):
            
            img_file = find_image_for_label(lbl_file_path, img_dir)
            if img_file is None:
                continue

            original_stem = os.path.splitext(os.path.basename(lbl_file_path))[0]
            new_stem = f"ds{ds_index}_{split}_{original_stem}"
            img_ext = os.path.splitext(img_file)[1]
            
            # Destination paths (canonical structure)
            dst_img = os.path.join("combined", split, "images", new_stem + img_ext)
            dst_lbl = os.path.join("combined", split, "labels", new_stem + ".txt")

            shutil.copy(img_file, dst_img)
            shutil.copy(lbl_file_path, dst_lbl)

def find_image_for_label(label_path, img_dir):
    """Finds the matching image for a given label file."""
    stem = os.path.splitext(os.path.basename(label_path))[0]
    exts = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
    for ext in exts:
        candidate = os.path.join(img_dir, stem + ext)
        if os.path.exists(candidate):
            return candidate
    return None

# Merge each dataset
for idx, path in enumerate(dataset_paths, start=1):
    merge_dataset(idx, path)
print("\nMerging complete.\n")


# FINAL YAML & TRAIN MODEL
combined_yaml = {
    "path": "./combined",
    "train": "train",
    "val": "valid",
    "test": "test",
    "names": [
        "toilet",              # 0
        "grab_handle",         # 1
        "mirror",              # 2
        "sink",                # 3
        "soap",                # 4
        "towel",               # 5 
        "wheelchair_logo",     # 6
        "door_handle",         # 7
        "emergency_button"     # 8
    ],
    "nc": 9
}

with open("combined.yaml", "w") as f:
    yaml.dump(combined_yaml, f, sort_keys=False) 

print("combined.yaml created successfully.\n")

model = YOLO("yolov8n.pt")

try:
    model.train(
        data="combined.yaml",
        epochs=50, 
        imgsz=640,
        batch=8,
        name="yolov8_combined_bathroom"
    )
    print("Training finished.")
except Exception as e:
    print(f"Training failed: {e}")

loading Roboflow workspace...
loading Roboflow project...


KeyboardInterrupt: 